In [0]:
base = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "silver/federal_elections/"
)

fed_df=spark.read.parquet(base)

In [0]:
print(fed_df.columns)
fed_df.printSchema()

['Wahlart', 'Wahltag', 'Gebietsart', 'Gebietsnummer', 'Gebietsname', 'UegGebietsart', 'UegGebietsnummer', 'Gruppenart', 'Gruppenname', 'Gruppenreihenfolge', 'Stimme', 'Anzahl', 'Prozent', 'VorpAnzahl', 'VorpProzent', 'DiffProzent', 'DiffProzentPkt', 'Gewählt', 'election_year']
root
 |-- Wahlart: string (nullable = true)
 |-- Wahltag: date (nullable = true)
 |-- Gebietsart: string (nullable = true)
 |-- Gebietsnummer: integer (nullable = true)
 |-- Gebietsname: string (nullable = true)
 |-- UegGebietsart: string (nullable = true)
 |-- UegGebietsnummer: integer (nullable = true)
 |-- Gruppenart: string (nullable = true)
 |-- Gruppenname: string (nullable = true)
 |-- Gruppenreihenfolge: integer (nullable = true)
 |-- Stimme: integer (nullable = true)
 |-- Anzahl: long (nullable = true)
 |-- Prozent: double (nullable = true)
 |-- VorpAnzahl: long (nullable = true)
 |-- VorpProzent: double (nullable = true)
 |-- DiffProzent: double (nullable = true)
 |-- DiffProzentPkt: double (nullable = 

In [0]:
from pyspark.sql import functions as F

gold_federal_results = fed_df.select(
    F.col("election_year").alias("wahljahr"),
    F.col("Wahlart").alias("wahltyp"),
    F.col("Wahltag").alias("wahltag"),

    F.col("Gebietsart").alias("gebietstyp"),
    F.col("Gebietsnummer").cast("string").alias("gebiet_id"),
    F.col("Gebietsname").alias("gebiet_name"),

    F.col("UegGebietsart").alias("uebergeordnetes_gebietstyp"),
    F.col("UegGebietsnummer").cast("string").alias("uebergeordnetes_gebiet_id"),

    F.col("Gruppenart").alias("gruppenart"),
    F.col("Gruppenname").alias("wahlvorschlag"),

    F.col("Stimme").alias("stimmenart"),
    F.col("Anzahl").alias("stimmen"),
    F.col("Prozent").alias("stimmen_pct"),

    F.col("VorpAnzahl").alias("vorperiode_stimmen"),
    F.col("VorpProzent").alias("vorperiode_pct"),
    F.col("DiffProzent").alias("veraenderung_pct"),
    F.col("DiffProzentPkt").alias("veraenderung_prozentpunkte"),

    F.col("Gewählt").alias("gewaehlt")
)

In [0]:
gold_federal_results.printSchema()

print(
    "Exact duplicates:",
    gold_federal_results.count()
    - gold_federal_results.distinct().count()
)

gold_federal_results.groupBy(
    "wahljahr",
    "gebietstyp",
    "stimmenart"
).count().orderBy(
    "wahljahr",
    "gebietstyp",
    "stimmenart"
).show(200, truncate=False)

root
 |-- wahljahr: integer (nullable = true)
 |-- wahltyp: string (nullable = true)
 |-- wahltag: date (nullable = true)
 |-- gebietstyp: string (nullable = true)
 |-- gebiet_id: string (nullable = true)
 |-- gebiet_name: string (nullable = true)
 |-- uebergeordnetes_gebietstyp: string (nullable = true)
 |-- uebergeordnetes_gebiet_id: string (nullable = true)
 |-- gruppenart: string (nullable = true)
 |-- wahlvorschlag: string (nullable = true)
 |-- stimmenart: integer (nullable = true)
 |-- stimmen: long (nullable = true)
 |-- stimmen_pct: double (nullable = true)
 |-- vorperiode_stimmen: long (nullable = true)
 |-- vorperiode_pct: double (nullable = true)
 |-- veraenderung_pct: double (nullable = true)
 |-- veraenderung_prozentpunkte: double (nullable = true)
 |-- gewaehlt: string (nullable = true)

Exact duplicates: 0
+--------+----------+----------+-----+
|wahljahr|gebietstyp|stimmenart|count|
+--------+----------+----------+-----+
|2021    |Bund      |NULL      |2    |
|2021    |

In [0]:
gold_federal_path = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "gold/election_results/federal/"
)

(
    gold_federal_results
    .write
    .mode("overwrite")
    .partitionBy("wahljahr")
    .parquet(gold_federal_path)
)

In [0]:
federal_summary = (
    gold_federal_results
    .filter(F.col("gebietstyp") == "Bund")
    .filter(F.col("stimmenart").isin(1, 2))
    .select(
        "wahljahr",
        "stimmenart",
        "wahlvorschlag",
        "stimmen",
        "stimmen_pct",
        "vorperiode_stimmen",
        "vorperiode_pct",
        "veraenderung_prozentpunkte"
    )
)

In [0]:
gold_federal_results.filter(
    F.col("gebietstyp") == "Bund"
).groupBy(
    "wahljahr",
    "stimmenart",
    "wahlvorschlag"
).count().filter(
    F.col("count") > 1
).show()

+--------+----------+-------------+-----+
|wahljahr|stimmenart|wahlvorschlag|count|
+--------+----------+-------------+-----+
+--------+----------+-------------+-----+



In [0]:
federal_summary.orderBy(
    "wahljahr",
    "stimmenart",
    F.desc("stimmen")
).show(100, truncate=False)

+--------+----------+---------------------------+--------+-----------+------------------+--------------+--------------------------+
|wahljahr|stimmenart|wahlvorschlag              |stimmen |stimmen_pct|vorperiode_stimmen|vorperiode_pct|veraenderung_prozentpunkte|
+--------+----------+---------------------------+--------+-----------+------------------+--------------+--------------------------+
|2021    |1         |Gültige                    |46362013|98.948884  |46389615          |98.751018     |0.197866                  |
|2021    |1         |SPD                        |12234690|26.389471  |11429231          |24.637478     |1.751993                  |
|2021    |1         |CDU                        |10451524|22.543292  |14030751          |30.245457     |-7.702165                 |
|2021    |1         |GRÜNE                      |6469081 |13.953408  |3717922           |8.014557      |5.938852                  |
|2021    |1         |AfD                        |4695611 |10.128143  |531749

In [0]:
federal_summary_path = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "gold/election_results/federal_summary/"
)

(
    federal_summary
    .write
    .mode("overwrite")
    .partitionBy("wahljahr")
    .parquet(federal_summary_path)
)